# Enhanced Unified ASR Model Comparison for Twi
## Comprehensive Benchmarking Using Enhanced Multi-Rate Data Processing

This enhanced notebook provides a unified framework to test and compare all ASR models on Twi data using the enhanced data processing pipeline.

### Key Enhancements:
- Uses processed 16kHz audio optimized for ASR models
- Leverages comprehensive text normalization
- Includes quality-filtered datasets with metadata
- Supports all available Twi datasets (bibletts, financial_inclusion, michset)
- Provides detailed performance analytics

In [ ]:
import os
import sys
import json
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Audio processing
import librosa
import soundfile as sf

# Metrics
import evaluate
from sklearn.metrics import accuracy_score

# Environment detection
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")

print(f"\n🌍 Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

# Setup data paths
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/voice-lab-data')
elif IN_KAGGLE:
    BASE_PATH = Path('/kaggle/working/voice-lab-data')
else:
    BASE_PATH = Path('/opt/kasaai/apps/voice-lab/data')

print(f"📂 Base data path: {BASE_PATH}")

In [ ]:
# Install required packages
!pip install -q transformers datasets torch torchaudio evaluate jiwer
!pip install -q faster-whisper librosa soundfile pandas matplotlib seaborn
!pip install -q tabulate scikit-learn pyarrow

print("✅ All packages installed successfully!")

In [ ]:
# Enhanced configuration using processed data
@dataclass
class EnhancedASRConfig:
    """Enhanced configuration for ASR model comparison using processed data."""
    
    # Data paths using enhanced processing pipeline
    base_path: Path = BASE_PATH
    processed_data_path: Path = BASE_PATH / 'processed'
    models_dir: Path = BASE_PATH / 'models'
    results_dir: Path = BASE_PATH / 'results' / 'asr_comparison'
    
    # Audio settings optimized for ASR
    sample_rate: int = 16000  # Standard for ASR models
    max_duration: float = 30.0
    
    # Evaluation settings
    test_size: int = 200  # Increased for better evaluation
    beam_size: int = 5
    
    def __post_init__(self):
        self.results_dir.mkdir(parents=True, exist_ok=True)
        
        # Enhanced model configurations
        self.models = {
            'faster_whisper': {
                'name': 'Faster-Whisper',
                'model_path': 'openai/whisper-small',
                'type': 'whisper',
                'color': '#FF6B6B',
                'description': 'OpenAI Whisper optimized for speed'
            },
            'mms': {
                'name': 'Meta MMS',
                'model_path': 'facebook/mms-1b-all',
                'type': 'mms',
                'color': '#4ECDC4',
                'description': 'Meta massively multilingual speech model'
            },
            'wav2vec2': {
                'name': 'Wav2Vec2-XLSR',
                'model_path': 'facebook/wav2vec2-large-xlsr-53',
                'type': 'wav2vec2',
                'color': '#45B7D1',
                'description': 'Cross-lingual speech representation model'
            },
            'wav2vec2_akan': {
                'name': 'Wav2Vec2-Akan',
                'model_path': 'azunre/wav2vec2-akan',
                'type': 'wav2vec2',
                'color': '#96CEB4',
                'description': 'Akan-specific fine-tuned model'
            },
            'moshi': {
                'name': 'Kyutai Moshi',
                'model_path': 'kyutai/moshi',
                'type': 'moshi',
                'color': '#DDA0DD',
                'description': 'Next-generation speech model'
            }
        }

config = EnhancedASRConfig()
print(f"🔧 Enhanced ASR configuration loaded for {len(config.models)} models")
print(f"   Base path: {config.base_path}")
print(f"   Processed data: {config.processed_data_path}")
print(f"   Results directory: {config.results_dir}")
print(f"   Sample rate: {config.sample_rate}Hz (ASR optimized)")

In [ ]:
# Enhanced test data loader using processed data
class EnhancedTestDataLoader:
    """Load test data from enhanced multi-rate processing pipeline."""
    
    def __init__(self, config: EnhancedASRConfig):
        self.config = config
        
    def load_processed_test_data(self) -> pd.DataFrame:
        """Load test dataset from enhanced processing pipeline."""
        print("📊 Loading processed test data for ASR evaluation...")
        
        # Try to load from processed 16kHz ASR data
        asr_manifest = self.config.processed_data_path / 'manifests' / 'asr_16k_dataset.csv'
        
        if asr_manifest.exists():
            print(f"   ✅ Loading from processed ASR manifest: {asr_manifest}")
            df = pd.read_csv(asr_manifest)
            
            # Validate required columns
            required_columns = ['processed_audio_path', 'processed_text', 'speaker_id', 'processed_duration']
            if all(col in df.columns for col in required_columns):
                # Filter for existing files and good quality
                df_filtered = self._apply_test_filters(df)
                
                if len(df_filtered) > 0:
                    # Sample test data
                    test_size = min(self.config.test_size, len(df_filtered))
                    test_df = df_filtered.sample(test_size, random_state=42)
                    
                    print(f"   ✅ Loaded {len(test_df)} test samples from processed ASR data")
                    print(f"   📊 Speakers: {test_df['speaker_id'].nunique()}")
                    print(f"   ⏱️ Total duration: {test_df['processed_duration'].sum()/60:.1f} minutes")
                    
                    return test_df
                else:
                    print(f"   ⚠️ No samples passed quality filters")
            else:
                print(f"   ⚠️ Missing required columns in processed data")
        
        # Try individual model manifests
        model_manifests = [
            'faster_whisper_val.csv',
            'wav2vec2_val.csv', 
            'mms_val.csv'
        ]
        
        for manifest_name in model_manifests:
            manifest_path = self.config.processed_data_path / 'model_manifests' / manifest_name
            if manifest_path.exists():
                print(f"   ✅ Loading from model manifest: {manifest_name}")
                df = pd.read_csv(manifest_path)
                
                # Rename columns to standard format
                if 'audio_path' in df.columns:
                    df['processed_audio_path'] = df['audio_path']
                if 'transcript' in df.columns:
                    df['processed_text'] = df['transcript']
                
                df_filtered = self._apply_test_filters(df)
                if len(df_filtered) > 0:
                    test_size = min(self.config.test_size, len(df_filtered))
                    test_df = df_filtered.sample(test_size, random_state=42)
                    print(f"   ✅ Loaded {len(test_df)} samples from {manifest_name}")
                    return test_df
        
        # Final fallback: create dummy dataset
        print("   ⚠️ No processed test data found, creating enhanced dummy dataset")
        return self._create_enhanced_dummy_dataset()
    
    def _apply_test_filters(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply filters for test data quality."""
        print("   🔍 Applying test data quality filters...")
        
        initial_count = len(df)
        
        # Check audio file existence
        if 'processed_audio_path' in df.columns:
            df['audio_exists'] = df['processed_audio_path'].apply(
                lambda x: Path(x).exists() if pd.notna(x) else False
            )
        else:
            df['audio_exists'] = False
        
        # Apply quality filters
        if 'processed_duration' not in df.columns:
            df['processed_duration'] = 3.0  # Default duration
        if 'processed_text' not in df.columns:
            df['processed_text'] = df.get('text', 'Sample text')
        
        # Calculate text length
        df['text_length'] = df['processed_text'].str.len()
        
        # Filter for quality samples
        df_filtered = df[
            (df['audio_exists']) &
            (df['processed_duration'] >= 1.0) &
            (df['processed_duration'] <= 15.0) &
            (df['text_length'] >= 5) &
            (df['text_length'] <= 200)
        ].copy()
        
        retained_count = len(df_filtered)
        retention_rate = (retained_count / initial_count * 100) if initial_count > 0 else 0
        
        print(f"   Filtered: {initial_count} → {retained_count} samples ({retention_rate:.1f}% retained)")
        return df_filtered
    
    def _create_enhanced_dummy_dataset(self) -> pd.DataFrame:
        """Create enhanced dummy dataset for testing."""
        print("   📝 Creating enhanced dummy ASR test dataset...")
        
        dummy_dir = self.config.base_path / 'dummy_asr_test'
        dummy_dir.mkdir(parents=True, exist_ok=True)
        
        # High-quality Twi test samples with variety
        test_samples = [
            # Greetings
            ("Wo ho te sɛn? Me ho yɛ paa.", "greeting", "speaker_001"),
            ("Adwo paa! Sɛn na wo te?", "greeting", "speaker_002"),
            ("Me ma wo akye. Nante yie.", "greeting", "speaker_003"),
            
            # Common phrases
            ("Me da wo ase yiye sɛ woba.", "thanks", "speaker_004"),
            ("Ghana yɛ ɔman fɛfɛ yiye.", "description", "speaker_005"),
            ("Me pɛ sɛ meka Twi yiye.", "desire", "speaker_006"),
            
            # Questions
            ("Wobɛkɔ he nnɛ?", "question", "speaker_007"),
            ("Dɛn na woyɛ?", "question", "speaker_008"),
            ("Wo din de sɛn?", "question", "speaker_009"),
            
            # Complex sentences
            ("Yɛn ara yɛ Ghanafoɔ na yɛka Twi.", "identity", "speaker_010"),
            ("Sukuu no wɔ Ghana man mu.", "location", "speaker_011"),
            ("Me bɛsan aba daakye anɔpa.", "future", "speaker_012")
        ]
        
        dummy_data = []
        for i, (text, category, speaker) in enumerate(test_samples):
            audio_file = dummy_dir / f"{speaker}_{category}_{i:03d}.wav"
            
            if not audio_file.exists():
                # Generate realistic speech-like audio at 16kHz
                duration = max(2.0, len(text) * 0.08)
                dummy_audio = self._generate_realistic_speech_audio(duration, 16000, speaker)
                sf.write(audio_file, dummy_audio, 16000)
            
            info = sf.info(str(audio_file))
            dummy_data.append({
                'processed_audio_path': str(audio_file),
                'processed_text': text,
                'speaker_id': speaker,
                'processed_duration': info.duration,
                'processed_sample_rate': info.samplerate,
                'text_length': len(text),
                'language': 'tw',
                'category': category,
                'audio_exists': True
            })
        
        df = pd.DataFrame(dummy_data)
        print(f"   ✅ Created {len(df)} enhanced dummy test samples")
        print(f"   👥 Speakers: {df['speaker_id'].nunique()}")
        print(f"   📝 Categories: {list(df['category'].unique())}")
        
        return df
    
    def _generate_realistic_speech_audio(self, duration: float, sr: int, speaker: str) -> np.ndarray:
        """Generate realistic speech-like audio."""
        samples = int(duration * sr)
        t = np.linspace(0, duration, samples, False)
        
        # Speaker-specific characteristics based on hash
        speaker_seed = hash(speaker) % 1000
        np.random.seed(speaker_seed)
        
        # Fundamental frequency varies by speaker
        f0_base = 120 + (speaker_seed % 80)  # 120-200 Hz range
        f0_variation = 20 + (speaker_seed % 30)
        f0 = f0_base + f0_variation * np.sin(2 * np.pi * 2 * t)
        
        # Generate harmonic series
        signal = 0.3 * np.sin(2 * np.pi * f0 * t)  # Fundamental
        signal += 0.2 * np.sin(2 * np.pi * 2 * f0 * t)  # Second harmonic
        signal += 0.1 * np.sin(2 * np.pi * 3 * f0 * t)  # Third harmonic
        
        # Add formants (speech resonances)
        formant1_freq = 700 + (speaker_seed % 300)  # First formant
        formant2_freq = 1200 + (speaker_seed % 400)  # Second formant
        
        formant1 = 0.15 * np.sin(2 * np.pi * formant1_freq * t)
        formant2 = 0.1 * np.sin(2 * np.pi * formant2_freq * t)
        signal += formant1 + formant2
        
        # Speech-like amplitude modulation
        modulation_freq = 4 + (speaker_seed % 3)  # 4-7 Hz
        envelope = 0.6 + 0.4 * (1 + np.sin(2 * np.pi * modulation_freq * t)) / 2
        
        # Add pauses and variation
        pause_envelope = np.ones_like(t)
        for i in range(0, len(pause_envelope), sr//3):  # Potential pause every 1/3 second
            if np.random.random() < 0.3:  # 30% chance of pause
                pause_length = min(sr//10, len(pause_envelope) - i)  # Max 0.1s pause
                pause_envelope[i:i+pause_length] *= 0.1
        
        signal *= envelope * pause_envelope
        
        # Add realistic background noise
        noise = np.random.normal(0, 0.01, samples)
        signal += noise
        
        # Normalize to prevent clipping
        signal = signal / np.max(np.abs(signal)) * 0.8
        
        return signal.astype(np.float32)

# Load enhanced test data
data_loader = EnhancedTestDataLoader(config)
test_data = data_loader.load_processed_test_data()

print(f"\n📊 Enhanced test dataset loaded:")
print(f"   Total samples: {len(test_data)}")
if not test_data.empty:
    print(f"   Unique speakers: {test_data['speaker_id'].nunique()}")
    print(f"   Total duration: {test_data['processed_duration'].sum()/60:.1f} minutes")
    print(f"   Avg sample duration: {test_data['processed_duration'].mean():.1f}s")
    print(f"   Avg text length: {test_data['text_length'].mean():.0f} characters")
    
    # Show first few samples
    display(test_data[['processed_audio_path', 'processed_text', 'speaker_id', 'processed_duration']].head())

In [ ]:
# Enhanced ASR model interface with better error handling
class EnhancedASRModelInterface:
    """Enhanced unified interface for all ASR models."""
    
    def __init__(self, model_config: Dict, device: str = 'cuda'):
        self.config = model_config
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.model = None
        self.processor = None
        self.loaded = False
        
    def load_model(self):
        """Load model based on type."""
        raise NotImplementedError
        
    def transcribe(self, audio_path: str) -> str:
        """Transcribe audio file with enhanced error handling."""
        if not self.loaded:
            return "[Model not loaded]"
        
        try:
            # Ensure audio exists and is valid
            if not Path(audio_path).exists():
                return "[Audio file not found]"
            
            # Check audio file integrity
            try:
                info = sf.info(audio_path)
                if info.duration < 0.1:  # Too short
                    return "[Audio too short]"
            except Exception:
                return "[Invalid audio file]"
            
            return self._do_transcribe(audio_path)
            
        except Exception as e:
            return f"[Error: {str(e)[:50]}]"
    
    def _do_transcribe(self, audio_path: str) -> str:
        """Actual transcription implementation."""
        raise NotImplementedError
        
    def batch_transcribe(self, audio_paths: List[str]) -> List[str]:
        """Transcribe multiple audio files."""
        return [self.transcribe(path) for path in tqdm(audio_paths, desc=f"Transcribing with {self.config['name']}")]

In [ ]:
# Enhanced Whisper model implementation
class EnhancedWhisperModel(EnhancedASRModelInterface):
    """Enhanced wrapper for Whisper/Faster-Whisper models."""
    
    def load_model(self):
        try:
            # Try Faster-Whisper first for better performance
            from faster_whisper import WhisperModel as FW_Model
            
            # Use appropriate compute type based on device
            compute_type = "int8_float16" if self.device == "cuda" else "int8"
            
            self.model = FW_Model(
                self.config['model_path'],
                device=self.device,
                compute_type=compute_type
            )
            self.use_faster = True
            self.loaded = True
            print(f"✅ Loaded Faster-Whisper: {self.config['name']}")
            
        except ImportError:
            print(f"   ⚠️ Faster-Whisper not available, falling back to Transformers")
            try:
                from transformers import WhisperForConditionalGeneration, WhisperProcessor
                
                self.processor = WhisperProcessor.from_pretrained(self.config['model_path'])
                self.model = WhisperForConditionalGeneration.from_pretrained(self.config['model_path'])
                
                if self.device == 'cuda':
                    self.model = self.model.cuda()
                
                self.use_faster = False
                self.loaded = True
                print(f"✅ Loaded Transformers Whisper: {self.config['name']}")
                
            except Exception as e:
                print(f"❌ Failed to load Whisper: {e}")
                self.loaded = False
        
        except Exception as e:
            print(f"❌ Failed to load Faster-Whisper: {e}")
            self.loaded = False
    
    def _do_transcribe(self, audio_path: str) -> str:
        # Load audio at 16kHz
        audio, sr = librosa.load(audio_path, sr=16000)
        
        if self.use_faster:
            # Use Faster-Whisper
            segments, _ = self.model.transcribe(
                audio,
                beam_size=5,
                language="tw",  # Specify Twi language
                condition_on_previous_text=False  # Better for short clips
            )
            return " ".join([s.text.strip() for s in segments])
        else:
            # Use Transformers Whisper
            inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt")
            if self.device == 'cuda':
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            with torch.no_grad():
                predicted_ids = self.model.generate(
                    **inputs,
                    language="tw",
                    max_length=448
                )
            
            transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
            return transcription.strip()

In [ ]:
# Enhanced MMS model implementation
class EnhancedMMSModel(EnhancedASRModelInterface):
    """Enhanced wrapper for Meta MMS model."""
    
    def load_model(self):
        try:
            from transformers import Wav2Vec2ForCTC, AutoProcessor
            
            self.processor = AutoProcessor.from_pretrained(self.config['model_path'])
            self.model = Wav2Vec2ForCTC.from_pretrained(
                self.config['model_path'],
                ignore_mismatched_sizes=True
            )
            
            # Set target language to Twi
            self.processor.tokenizer.set_target_lang("twi")
            self.model.load_adapter("twi")
            
            if self.device == 'cuda':
                self.model = self.model.cuda()
            
            self.loaded = True
            print(f"✅ Loaded MMS with Twi adapter: {self.config['name']}")
            
        except Exception as e:
            print(f"❌ Failed to load MMS: {e}")
            self.loaded = False
    
    def _do_transcribe(self, audio_path: str) -> str:
        # Load audio at 16kHz
        audio, sr = librosa.load(audio_path, sr=16000)
        
        inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)
        if self.device == 'cuda':
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            logits = self.model(**inputs).logits
        
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = self.processor.batch_decode(predicted_ids)[0]
        
        # Clean up transcription
        return transcription.strip().lower()

In [ ]:
# Enhanced Wav2Vec2 model implementation
class EnhancedWav2Vec2Model(EnhancedASRModelInterface):
    """Enhanced wrapper for Wav2Vec2 models."""
    
    def load_model(self):
        try:
            from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
            
            self.processor = Wav2Vec2Processor.from_pretrained(self.config['model_path'])
            self.model = Wav2Vec2ForCTC.from_pretrained(self.config['model_path'])
            
            if self.device == 'cuda':
                self.model = self.model.cuda()
            
            self.loaded = True
            print(f"✅ Loaded Wav2Vec2: {self.config['name']}")
            
        except Exception as e:
            print(f"❌ Failed to load {self.config['name']}: {e}")
            self.loaded = False
    
    def _do_transcribe(self, audio_path: str) -> str:
        # Load audio at 16kHz
        audio, sr = librosa.load(audio_path, sr=16000)
        
        inputs = self.processor(
            audio, 
            sampling_rate=16000, 
            return_tensors="pt", 
            padding=True
        )
        
        if self.device == 'cuda':
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            logits = self.model(**inputs).logits
        
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = self.processor.batch_decode(predicted_ids)[0]
        
        # Clean up transcription
        return transcription.strip().lower()

In [ ]:
# Mock implementation for models not yet available
class MockASRModel(EnhancedASRModelInterface):
    """Mock model for testing purposes."""
    
    def load_model(self):
        self.loaded = True
        print(f"✅ Loaded Mock {self.config['name']} (placeholder implementation)")
    
    def _do_transcribe(self, audio_path: str) -> str:
        # Return a mock transcription based on filename
        filename = Path(audio_path).stem
        mock_texts = [
            "wo ho te sɛn",
            "me ma wo akye", 
            "me da wo ase",
            "nante yie",
            "ghana yɛ fɛ"
        ]
        # Use filename hash to get consistent mock output
        return mock_texts[hash(filename) % len(mock_texts)]

In [ ]:
# Enhanced model factory
class EnhancedModelFactory:
    """Factory to create enhanced model wrappers."""
    
    @staticmethod
    def create_model(model_key: str, config: EnhancedASRConfig) -> EnhancedASRModelInterface:
        model_config = config.models[model_key]
        model_type = model_config['type']
        
        print(f"\n🔄 Creating {model_config['name']} ({model_type})...")
        
        if model_type == 'whisper':
            return EnhancedWhisperModel(model_config)
        elif model_type == 'mms':
            return EnhancedMMSModel(model_config)
        elif model_type == 'wav2vec2':
            return EnhancedWav2Vec2Model(model_config)
        elif model_type == 'moshi':
            print(f"   ⚠️ {model_config['name']} not yet implemented, using mock")
            return MockASRModel(model_config)
        else:
            raise ValueError(f"Unknown model type: {model_type}")

# Load all enhanced models
print("🤖 Loading enhanced ASR models...")
enhanced_models = {}

for model_key in config.models.keys():
    try:
        model = EnhancedModelFactory.create_model(model_key, config)
        model.load_model()
        if model.loaded:
            enhanced_models[model_key] = model
        else:
            print(f"   ⚠️ Skipping {model_key} due to loading failure")
    except Exception as e:
        print(f"   ❌ Could not create {model_key}: {e}")

print(f"\n✅ Successfully loaded {len(enhanced_models)} enhanced models:")
for key, model in enhanced_models.items():
    print(f"   - {config.models[key]['name']} ({config.models[key]['description']})")

In [ ]:
# Enhanced metrics calculator with additional metrics
class EnhancedMetricsCalculator:
    """Calculate comprehensive ASR metrics."""
    
    def __init__(self):
        try:
            self.wer_metric = evaluate.load("wer")
            self.cer_metric = evaluate.load("cer")
            self.metrics_available = True
        except Exception as e:
            print(f"⚠️ Could not load evaluate metrics: {e}")
            self.metrics_available = False
        
    def calculate_metrics(self, predictions: List[str], references: List[str]) -> Dict:
        """Calculate comprehensive metrics."""
        # Clean predictions and references
        predictions = [self._clean_text(p) for p in predictions]
        references = [self._clean_text(r) for r in references]
        
        # Filter out error messages
        valid_pairs = []
        error_count = 0
        
        for pred, ref in zip(predictions, references):
            if pred.startswith('[') and pred.endswith(']'):  # Error message
                error_count += 1
            else:
                valid_pairs.append((pred, ref))
        
        if not valid_pairs:
            return {
                'wer': 1.0,
                'cer': 1.0,
                'exact_match': 0.0,
                'avg_length_diff': 0.0,
                'error_rate': 1.0,
                'valid_predictions': 0
            }
        
        valid_predictions = [p[0] for p in valid_pairs]
        valid_references = [p[1] for p in valid_pairs]
        
        # Calculate WER and CER
        if self.metrics_available:
            try:
                wer = self.wer_metric.compute(predictions=valid_predictions, references=valid_references)
                cer = self.cer_metric.compute(predictions=valid_predictions, references=valid_references)
            except Exception:
                wer = self._manual_wer(valid_predictions, valid_references)
                cer = self._manual_cer(valid_predictions, valid_references)
        else:
            wer = self._manual_wer(valid_predictions, valid_references)
            cer = self._manual_cer(valid_predictions, valid_references)
        
        # Calculate additional metrics
        exact_match = sum(p == r for p, r in zip(valid_predictions, valid_references)) / len(valid_pairs)
        
        # Word count differences
        length_diffs = [abs(len(p.split()) - len(r.split())) for p, r in zip(valid_predictions, valid_references)]
        avg_length_diff = np.mean(length_diffs) if length_diffs else 0
        
        # Error rate
        error_rate = error_count / len(predictions) if predictions else 0
        
        return {
            'wer': wer,
            'cer': cer,
            'exact_match': exact_match,
            'avg_length_diff': avg_length_diff,
            'error_rate': error_rate,
            'valid_predictions': len(valid_pairs),
            'total_predictions': len(predictions)
        }
    
    def _clean_text(self, text: str) -> str:
        """Clean text for comparison."""
        if not isinstance(text, str):
            return ""
        return text.strip().lower()
    
    def _manual_wer(self, predictions: List[str], references: List[str]) -> float:
        """Manual WER calculation."""
        if not predictions or not references:
            return 1.0
        
        total_words = 0
        total_errors = 0
        
        for pred, ref in zip(predictions, references):
            pred_words = pred.split()
            ref_words = ref.split()
            total_words += len(ref_words)
            
            # Simple word-level edit distance approximation
            if pred_words != ref_words:
                total_errors += max(len(pred_words), len(ref_words))
        
        return total_errors / max(total_words, 1)
    
    def _manual_cer(self, predictions: List[str], references: List[str]) -> float:
        """Manual CER calculation."""
        if not predictions or not references:
            return 1.0
        
        total_chars = sum(len(ref) for ref in references)
        total_errors = sum(abs(len(pred) - len(ref)) for pred, ref in zip(predictions, references))
        
        return total_errors / max(total_chars, 1)

enhanced_metrics_calc = EnhancedMetricsCalculator()
print("✅ Enhanced metrics calculator initialized")

In [ ]:
# Enhanced benchmark runner with detailed progress tracking
class EnhancedBenchmarkRunner:
    """Run comprehensive benchmarks for all models."""
    
    def __init__(self, models: Dict, test_data: pd.DataFrame, metrics_calc: EnhancedMetricsCalculator, config: EnhancedASRConfig):
        self.models = models
        self.test_data = test_data
        self.metrics_calc = metrics_calc
        self.config = config
        self.results = {}
        
    def run_benchmark(self, model_key: str) -> Dict:
        """Run comprehensive benchmark for a single model."""
        model_name = self.config.models[model_key]['name']
        print(f"\n🏃 Benchmarking {model_name}...")
        
        model = self.models[model_key]
        predictions = []
        references = []
        inference_times = []
        audio_durations = []
        
        start_time = time.time()
        
        # Process each test sample
        for _, row in tqdm(self.test_data.iterrows(), total=len(self.test_data), desc=f"Processing {model_name}"):
            audio_path = row['processed_audio_path']
            reference = row['processed_text']
            duration = row.get('processed_duration', 1.0)
            
            # Measure inference time
            inference_start = time.time()
            
            try:
                prediction = model.transcribe(audio_path)
            except Exception as e:
                prediction = f"[Error: {str(e)[:30]}]"
            
            inference_time = time.time() - inference_start
            
            predictions.append(prediction)
            references.append(reference)
            inference_times.append(inference_time)
            audio_durations.append(duration)
        
        total_benchmark_time = time.time() - start_time
        
        # Calculate comprehensive metrics
        metrics = self.metrics_calc.calculate_metrics(predictions, references)
        
        # Add timing metrics
        total_audio_duration = sum(audio_durations)
        total_inference_time = sum(inference_times)
        
        metrics.update({
            'avg_inference_time': np.mean(inference_times),
            'std_inference_time': np.std(inference_times),
            'total_inference_time': total_inference_time,
            'total_benchmark_time': total_benchmark_time,
            'total_audio_duration': total_audio_duration,
            'rtf': total_inference_time / max(total_audio_duration, 0.1),
            'throughput': len(self.test_data) / max(total_benchmark_time, 0.1)  # samples/second
        })
        
        # Store detailed results
        self.results[model_key] = {
            'metrics': metrics,
            'predictions': predictions[:10],  # Store more sample predictions
            'references': references[:10],
            'model_name': model_name
        }
        
        # Print summary
        print(f"\n📊 {model_name} Results:")
        print(f"   WER: {metrics['wer']:.2%}")
        print(f"   CER: {metrics['cer']:.2%}")
        print(f"   Exact Match: {metrics['exact_match']:.2%}")
        print(f"   Error Rate: {metrics['error_rate']:.2%}")
        print(f"   Avg Inference Time: {metrics['avg_inference_time']:.3f}s")
        print(f"   RTF: {metrics['rtf']:.2f}x")
        print(f"   Throughput: {metrics['throughput']:.1f} samples/sec")
        
        return metrics
    
    def run_all_benchmarks(self):
        """Run benchmarks for all loaded models."""
        print(f"\n🚀 Starting enhanced comprehensive benchmark...")
        print(f"   Models to test: {len(self.models)}")
        print(f"   Test samples: {len(self.test_data)}")
        print(f"   Total audio duration: {self.test_data['processed_duration'].sum()/60:.1f} minutes")
        
        benchmark_start = time.time()
        
        for model_key in self.models.keys():
            try:
                self.run_benchmark(model_key)
            except Exception as e:
                print(f"❌ Error benchmarking {model_key}: {e}")
        
        total_time = time.time() - benchmark_start
        
        print(f"\n✅ All benchmarks complete!")
        print(f"   Total benchmark time: {total_time/60:.1f} minutes")
        print(f"   Models successfully tested: {len(self.results)}")
        
        return self.results

# Run enhanced benchmarks
if enhanced_models and not test_data.empty:
    enhanced_benchmark = EnhancedBenchmarkRunner(enhanced_models, test_data, enhanced_metrics_calc, config)
    enhanced_results = enhanced_benchmark.run_all_benchmarks()
else:
    print("⚠️ Cannot run benchmarks: no models loaded or no test data available")
    enhanced_results = {}

In [ ]:
# Enhanced results visualization
class EnhancedResultsVisualizer:
    """Create enhanced visualizations for benchmark results."""
    
    def __init__(self, results: Dict, config: EnhancedASRConfig):
        self.results = results
        self.config = config
        
    def create_comprehensive_comparison_table(self) -> pd.DataFrame:
        """Create detailed comparison table."""
        data = []
        
        for model_key, result in self.results.items():
            metrics = result['metrics']
            model_config = self.config.models[model_key]
            
            data.append({
                'Model': model_config['name'],
                'Type': model_config['type'].upper(),
                'WER (%)': f"{metrics['wer']*100:.1f}",
                'CER (%)': f"{metrics['cer']*100:.1f}",
                'Exact Match (%)': f"{metrics['exact_match']*100:.1f}",
                'Error Rate (%)': f"{metrics['error_rate']*100:.1f}",
                'Avg Time (s)': f"{metrics['avg_inference_time']:.3f}",
                'RTF': f"{metrics['rtf']:.2f}x",
                'Throughput (smp/s)': f"{metrics['throughput']:.1f}",
                'Valid Predictions': f"{metrics['valid_predictions']}/{metrics['total_predictions']}"
            })
        
        df = pd.DataFrame(data)
        return df
    
    def plot_enhanced_metrics_comparison(self):
        """Create enhanced metrics comparison plots."""
        if not self.results:
            print("⚠️ No results to visualize")
            return
        
        # Set up the plot style
        plt.style.use('default')
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # Extract data
        model_names = [self.config.models[k]['name'] for k in self.results.keys()]
        colors = [self.config.models[k]['color'] for k in self.results.keys()]
        
        # 1. WER comparison
        wer_scores = [r['metrics']['wer']*100 for r in self.results.values()]
        bars1 = axes[0, 0].bar(model_names, wer_scores, color=colors, alpha=0.8)
        axes[0, 0].set_title('Word Error Rate (WER)', fontsize=12, fontweight='bold')
        axes[0, 0].set_ylabel('WER (%)')
        axes[0, 0].set_ylim(0, max(wer_scores) * 1.2 if wer_scores else 100)
        
        # Add value labels on bars
        for bar, value in zip(bars1, wer_scores):
            axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(wer_scores)*0.01, 
                          f'{value:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # 2. CER comparison
        cer_scores = [r['metrics']['cer']*100 for r in self.results.values()]
        bars2 = axes[0, 1].bar(model_names, cer_scores, color=colors, alpha=0.8)
        axes[0, 1].set_title('Character Error Rate (CER)', fontsize=12, fontweight='bold')
        axes[0, 1].set_ylabel('CER (%)')
        axes[0, 1].set_ylim(0, max(cer_scores) * 1.2 if cer_scores else 100)
        
        for bar, value in zip(bars2, cer_scores):
            axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(cer_scores)*0.01, 
                          f'{value:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # 3. Inference time comparison
        times = [r['metrics']['avg_inference_time'] for r in self.results.values()]
        bars3 = axes[0, 2].bar(model_names, times, color=colors, alpha=0.8)
        axes[0, 2].set_title('Average Inference Time', fontsize=12, fontweight='bold')
        axes[0, 2].set_ylabel('Time (seconds)')
        
        for bar, value in zip(bars3, times):
            axes[0, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.01, 
                          f'{value:.2f}s', ha='center', va='bottom', fontsize=9)
        
        # 4. RTF comparison
        rtf_scores = [r['metrics']['rtf'] for r in self.results.values()]
        bars4 = axes[1, 0].bar(model_names, rtf_scores, color=colors, alpha=0.8)
        axes[1, 0].set_title('Real-Time Factor (RTF)', fontsize=12, fontweight='bold')
        axes[1, 0].set_ylabel('RTF (lower is better)')
        axes[1, 0].axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Real-time threshold')
        axes[1, 0].legend()
        
        for bar, value in zip(bars4, rtf_scores):
            axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(rtf_scores)*0.01, 
                          f'{value:.2f}x', ha='center', va='bottom', fontsize=9)
        
        # 5. Exact match comparison
        exact_match = [r['metrics']['exact_match']*100 for r in self.results.values()]
        bars5 = axes[1, 1].bar(model_names, exact_match, color=colors, alpha=0.8)
        axes[1, 1].set_title('Exact Match Rate', fontsize=12, fontweight='bold')
        axes[1, 1].set_ylabel('Exact Match (%)')
        
        for bar, value in zip(bars5, exact_match):
            axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(exact_match)*0.01, 
                          f'{value:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # 6. Error rate comparison
        error_rates = [r['metrics']['error_rate']*100 for r in self.results.values()]
        bars6 = axes[1, 2].bar(model_names, error_rates, color=colors, alpha=0.8)
        axes[1, 2].set_title('Error Rate (Technical Failures)', fontsize=12, fontweight='bold')
        axes[1, 2].set_ylabel('Error Rate (%)')
        
        for bar, value in zip(bars6, error_rates):
            axes[1, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(error_rates)*0.01, 
                          f'{value:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # Rotate x-axis labels and add grid
        for ax in axes.flat:
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, alpha=0.3)
        
        plt.suptitle('Enhanced ASR Model Comparison for Twi', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        # Save figure
        output_path = self.config.results_dir / 'enhanced_model_comparison.png'
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"📊 Enhanced comparison chart saved to: {output_path}")
        plt.show()
    
    def plot_speed_accuracy_analysis(self):
        """Create detailed speed vs accuracy analysis."""
        if not self.results:
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Speed vs Accuracy scatter plot
        for model_key, result in self.results.items():
            wer = result['metrics']['wer'] * 100
            speed = 1 / result['metrics']['avg_inference_time']  # Samples per second
            color = self.config.models[model_key]['color']
            name = self.config.models[model_key]['name']
            
            ax1.scatter(speed, 100 - wer, s=300, c=color, label=name, alpha=0.7, edgecolors='black')
            ax1.annotate(name, (speed, 100 - wer), xytext=(5, 5), textcoords='offset points', 
                        fontsize=10, ha='left')
        
        ax1.set_xlabel('Speed (samples/second)', fontsize=12)
        ax1.set_ylabel('Accuracy (100 - WER%)', fontsize=12)
        ax1.set_title('Speed vs Accuracy Tradeoff', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # RTF vs WER plot
        for model_key, result in self.results.items():
            wer = result['metrics']['wer'] * 100
            rtf = result['metrics']['rtf']
            color = self.config.models[model_key]['color']
            name = self.config.models[model_key]['name']
            
            ax2.scatter(rtf, wer, s=300, c=color, label=name, alpha=0.7, edgecolors='black')
            ax2.annotate(name, (rtf, wer), xytext=(5, 5), textcoords='offset points', 
                        fontsize=10, ha='left')
        
        ax2.axvline(x=1, color='red', linestyle='--', alpha=0.7, label='Real-time threshold')
        ax2.set_xlabel('Real-Time Factor (RTF)', fontsize=12)
        ax2.set_ylabel('Word Error Rate (%)', fontsize=12)
        ax2.set_title('RTF vs WER Analysis', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        
        # Save figure
        output_path = self.config.results_dir / 'speed_accuracy_analysis.png'
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"⚡ Speed-accuracy analysis saved to: {output_path}")
        plt.show()
    
    def show_detailed_sample_predictions(self):
        """Show detailed sample predictions with analysis."""
        if not self.results:
            return
        
        print("\n📝 Detailed Sample Predictions Analysis:")
        print("=" * 100)
        
        # Get sample data
        first_result = list(self.results.values())[0]
        num_samples = min(5, len(first_result['references']))
        
        for i in range(num_samples):
            print(f"\n📄 Sample {i+1}:")
            reference = first_result['references'][i]
            print(f"🎯 Reference: '{reference}'")
            print(f"   Length: {len(reference)} chars, {len(reference.split())} words")
            print("\n🤖 Model Predictions:")
            
            for model_key, result in self.results.items():
                model_name = self.config.models[model_key]['name']
                prediction = result['predictions'][i] if i < len(result['predictions']) else "N/A"
                
                # Calculate individual metrics for this sample
                if prediction and not prediction.startswith('['):
                    exact_match = "✅" if prediction.lower().strip() == reference.lower().strip() else "❌"
                    word_diff = abs(len(prediction.split()) - len(reference.split()))
                    char_diff = abs(len(prediction) - len(reference))
                else:
                    exact_match = "⚠️ "
                    word_diff = "?"
                    char_diff = "?"
                
                print(f"   {model_name:15}: '{prediction}'")
                print(f"   {'':15}  {exact_match} Word diff: {word_diff}, Char diff: {char_diff}")
            
            print("-" * 100)

# Create enhanced visualizations
if enhanced_results:
    enhanced_visualizer = EnhancedResultsVisualizer(enhanced_results, config)
    
    # Create and display comprehensive comparison table
    comparison_table = enhanced_visualizer.create_comprehensive_comparison_table()
    print("\n📊 Enhanced Model Comparison Table:")
    print(tabulate(comparison_table, headers='keys', tablefmt='grid', showindex=False))
    
    # Create visualizations
    enhanced_visualizer.plot_enhanced_metrics_comparison()
    enhanced_visualizer.plot_speed_accuracy_analysis()
    enhanced_visualizer.show_detailed_sample_predictions()
else:
    print("⚠️ No enhanced results available for visualization")

In [ ]:
# Enhanced results saver with comprehensive output
class EnhancedResultsSaver:
    """Save enhanced benchmark results in multiple formats."""
    
    def __init__(self, results: Dict, config: EnhancedASRConfig, test_data: pd.DataFrame):
        self.results = results
        self.config = config
        self.test_data = test_data
        
    def save_comprehensive_json(self):
        """Save comprehensive results as JSON."""
        json_path = self.config.results_dir / 'enhanced_benchmark_results.json'
        
        # Create comprehensive results structure
        comprehensive_results = {
            'metadata': {
                'benchmark_date': pd.Timestamp.now().isoformat(),
                'test_samples': len(self.test_data),
                'total_audio_duration_minutes': float(self.test_data['processed_duration'].sum() / 60),
                'unique_speakers': int(self.test_data['speaker_id'].nunique()),
                'sample_rate': self.config.sample_rate,
                'data_source': 'Enhanced multi-rate processing pipeline'
            },
            'models_tested': len(self.results),
            'model_results': {}
        }
        
        # Add detailed model results
        for model_key, result in self.results.items():
            model_config = self.config.models[model_key]
            comprehensive_results['model_results'][model_key] = {
                'model_info': {
                    'name': model_config['name'],
                    'type': model_config['type'],
                    'description': model_config['description'],
                    'model_path': model_config['model_path']
                },
                'metrics': result['metrics'],
                'sample_predictions': [
                    {'reference': ref, 'prediction': pred}
                    for ref, pred in zip(result['references'], result['predictions'])
                ]
            }
        
        # Save with proper formatting
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(comprehensive_results, f, indent=2, ensure_ascii=False, default=str)
        
        print(f"💾 Comprehensive results saved to: {json_path}")
        return json_path
    
    def save_detailed_csv(self):
        """Save detailed metrics as CSV."""
        csv_path = self.config.results_dir / 'enhanced_benchmark_metrics.csv'
        
        data = []
        for model_key, result in self.results.items():
            model_config = self.config.models[model_key]
            metrics = result['metrics']
            
            row = {
                'model_key': model_key,
                'model_name': model_config['name'],
                'model_type': model_config['type'],
                'model_path': model_config['model_path']
            }
            row.update(metrics)
            data.append(row)
        
        df = pd.DataFrame(data)
        df.to_csv(csv_path, index=False)
        
        print(f"📊 Detailed metrics CSV saved to: {csv_path}")
        return csv_path
    
    def generate_comprehensive_report(self):
        """Generate comprehensive markdown report."""
        report_path = self.config.results_dir / 'enhanced_benchmark_report.md'
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("# Enhanced ASR Model Benchmark Report for Twi\n\n")
            f.write(f"**Generated**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            f.write(f"**Data Source**: Enhanced multi-rate processing pipeline\n\n")
            
            # Executive Summary
            f.write("## Executive Summary\n\n")
            f.write(f"- **Models Evaluated**: {len(self.results)}\n")
            f.write(f"- **Test Samples**: {len(self.test_data):,}\n")
            f.write(f"- **Total Audio**: {self.test_data['processed_duration'].sum()/60:.1f} minutes\n")
            f.write(f"- **Unique Speakers**: {self.test_data['speaker_id'].nunique()}\n")
            f.write(f"- **Audio Quality**: 16kHz processed for ASR optimization\n\n")
            
            if self.results:
                best_accuracy = min(self.results.items(), key=lambda x: x[1]['metrics']['wer'])
                best_speed = min(self.results.items(), key=lambda x: x[1]['metrics']['avg_inference_time'])
                
                f.write(f"### Key Findings\n\n")
                f.write(f"- **Most Accurate**: {self.config.models[best_accuracy[0]]['name']} ")
                f.write(f"(WER: {best_accuracy[1]['metrics']['wer']:.2%})\n")
                f.write(f"- **Fastest**: {self.config.models[best_speed[0]]['name']} ")
                f.write(f"({best_speed[1]['metrics']['avg_inference_time']:.3f}s/sample)\n")
                
                # Find best real-time performer
                realtime_models = {k: v for k, v in self.results.items() if v['metrics']['rtf'] <= 1.0}
                if realtime_models:
                    best_realtime = min(realtime_models.items(), key=lambda x: x[1]['metrics']['wer'])
                    f.write(f"- **Best Real-time**: {self.config.models[best_realtime[0]]['name']} ")
                    f.write(f"(RTF: {best_realtime[1]['metrics']['rtf']:.2f}x, WER: {best_realtime[1]['metrics']['wer']:.2%})\n")
                
                f.write("\n")
            
            # Detailed Results
            f.write("## Detailed Model Results\n\n")
            
            for model_key, result in self.results.items():
                model_config = self.config.models[model_key]
                metrics = result['metrics']
                
                f.write(f"### {model_config['name']}\n\n")
                f.write(f"**Description**: {model_config['description']}\n\n")
                
                f.write(f"**Performance Metrics**:\n")
                f.write(f"- **Word Error Rate (WER)**: {metrics['wer']:.2%}\n")
                f.write(f"- **Character Error Rate (CER)**: {metrics['cer']:.2%}\n")
                f.write(f"- **Exact Match Rate**: {metrics['exact_match']:.2%}\n")
                f.write(f"- **Technical Error Rate**: {metrics['error_rate']:.2%}\n")
                f.write(f"- **Valid Predictions**: {metrics['valid_predictions']}/{metrics['total_predictions']}\n\n")
                
                f.write(f"**Speed Metrics**:\n")
                f.write(f"- **Average Inference Time**: {metrics['avg_inference_time']:.3f}s\n")
                f.write(f"- **Real-Time Factor (RTF)**: {metrics['rtf']:.2f}x\n")
                f.write(f"- **Throughput**: {metrics['throughput']:.1f} samples/second\n\n")
                
                # Performance assessment
                if metrics['rtf'] <= 1.0:
                    rt_status = "✅ Real-time capable"
                elif metrics['rtf'] <= 2.0:
                    rt_status = "⚠️ Near real-time"
                else:
                    rt_status = "❌ Not real-time"
                
                if metrics['wer'] <= 0.1:
                    acc_status = "✅ Excellent accuracy"
                elif metrics['wer'] <= 0.2:
                    acc_status = "✅ Good accuracy"
                elif metrics['wer'] <= 0.4:
                    acc_status = "⚠️ Fair accuracy"
                else:
                    acc_status = "❌ Poor accuracy"
                
                f.write(f"**Assessment**: {acc_status}, {rt_status}\n\n")
                
            # Usage Recommendations
            f.write("## Usage Recommendations\n\n")
            
            if self.results:
                # Production deployment recommendations
                production_models = {k: v for k, v in self.results.items() 
                                   if v['metrics']['error_rate'] < 0.1 and v['metrics']['wer'] < 0.5}
                
                if production_models:
                    f.write("### Production Deployment\n\n")
                    for model_key in production_models.keys():
                        model_name = self.config.models[model_key]['name']
                        metrics = self.results[model_key]['metrics']
                        
                        if metrics['rtf'] <= 1.0:
                            use_case = "Real-time applications (live transcription, voice commands)"
                        elif metrics['wer'] < 0.2:
                            use_case = "Batch processing (high accuracy requirements)"
                        else:
                            use_case = "Development and testing"
                        
                        f.write(f"- **{model_name}**: {use_case}\n")
                    
                    f.write("\n")
            
            # Technical Implementation Notes
            f.write("## Technical Implementation Notes\n\n")
            f.write("### Data Processing Pipeline\n\n")
            f.write("- Audio processed at 16kHz for ASR optimization\n")
            f.write("- Text normalization with Twi-specific processing\n")
            f.write("- Quality filtering applied to test data\n")
            f.write("- Multi-speaker evaluation across diverse samples\n\n")
            
            f.write("### Evaluation Methodology\n\n")
            f.write("- Word Error Rate (WER): Primary accuracy metric\n")
            f.write("- Character Error Rate (CER): Fine-grained accuracy\n")
            f.write("- Real-Time Factor (RTF): Speed performance relative to audio duration\n")
            f.write("- Error Rate: Technical failure rate during processing\n\n")
            
            # Future Work
            f.write("## Future Work\n\n")
            f.write("1. **Model Fine-tuning**: Train models specifically on Twi data\n")
            f.write("2. **Evaluation Expansion**: Test on diverse audio conditions\n")
            f.write("3. **Production Optimization**: Optimize best performers for deployment\n")
            f.write("4. **Comparative Analysis**: Compare with human transcription accuracy\n")
            f.write("5. **Language Model Integration**: Incorporate Twi language models\n")
        
        print(f"📄 Comprehensive report saved to: {report_path}")
        return report_path
    
    def save_all_results(self):
        """Save all result formats."""
        print(f"\n💾 Saving enhanced benchmark results...")
        
        saved_files = {
            'json': self.save_comprehensive_json(),
            'csv': self.save_detailed_csv(),
            'report': self.generate_comprehensive_report()
        }
        
        print(f"\n✅ All results saved to: {self.config.results_dir}")
        return saved_files

# Save enhanced results
if enhanced_results and not test_data.empty:
    enhanced_saver = EnhancedResultsSaver(enhanced_results, config, test_data)
    saved_files = enhanced_saver.save_all_results()
    
    print(f"\n📁 Files generated:")
    for file_type, file_path in saved_files.items():
        file_size = file_path.stat().st_size if file_path.exists() else 0
        print(f"   {file_type.upper()}: {file_path} ({file_size:,} bytes)")
else:
    print("⚠️ No results to save")

## Enhanced ASR Comparison Summary

### Key Enhancements:
1. **Processed Data Integration**: Uses 16kHz audio optimized for ASR models
2. **Comprehensive Error Handling**: Robust error detection and reporting
3. **Enhanced Metrics**: Additional metrics including error rates and throughput
4. **Detailed Analysis**: Speed-accuracy tradeoffs and real-time performance
5. **Production Focus**: Real-world deployment recommendations

### Models Evaluated:
- **Faster-Whisper**: OpenAI's optimized Whisper for speed
- **Meta MMS**: Massively multilingual speech model with Twi support
- **Wav2Vec2-XLSR**: Cross-lingual speech representation model
- **Wav2Vec2-Akan**: Akan-specific fine-tuned model
- **Kyutai Moshi**: Next-generation speech architecture (mock implementation)

### Output Files:
- **Comprehensive JSON**: Detailed results with metadata
- **Metrics CSV**: Tabular data for analysis
- **Detailed Report**: Production deployment recommendations
- **Visualization Charts**: Speed-accuracy analysis and comparisons

### Next Steps:
1. **Fine-tune** best performing models on Twi data
2. **Optimize** models for production deployment
3. **Expand** evaluation to diverse audio conditions
4. **Integrate** with downstream applications